# Day 3 - In-Class Assignment 3

## Multi-Class News Classification with DistilBERT

**Suggested time:** final 60 minutes of class  
**Total marks:** 30

This notebook is intentionally scaffolded but does **not** contain answer code. Use the concepts and code patterns from the main Day 3 learning notebook.

### Dataset
You will use **AG News**, a real multi-class news-classification dataset with four labels:

- World
- Sports
- Business
- Sci/Tech

### Questions

- **Q1 [10 marks]** Fine-tune DistilBERT and report macro F1.
- **Q2 [10 marks]** Visualize attention for one correct and one wrong prediction using BertViz and explain what you observe.
- **Q3 [10 marks]** Compare BERT and RoBERTa tokenization on five sentences and summarize the differences.

## 0. Setup

Use a Colab GPU if available:

**Runtime -> Change runtime type -> T4 GPU**

In [ ]:
%pip install -q -U transformers datasets accelerate scikit-learn bertviz

In [ ]:

import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# Q1 [10 marks] - Fine-tune DistilBERT on AG News

## Task

1. Load the AG News dataset.
2. Inspect the label names and a few examples.
3. Load `distilbert-base-uncased` using `AutoTokenizer`.
4. Tokenize the text using truncation.
5. Use dynamic padding.
6. Load `AutoModelForSequenceClassification` with 4 labels.
7. Fine-tune the model.
8. Evaluate it using **macro precision, macro recall and macro F1**.
9. Report your final macro F1.

### Why macro F1?
Macro F1 calculates F1 separately for each class and gives every class equal importance in the final average.

In [ ]:

# Dataset setup - provided so you can spend more time on modeling.

ag_news = load_dataset("fancyzhx/ag_news")

label_names = ag_news["train"].features["label"].names
print("Labels:", label_names)
print("Example:", ag_news["train"][0])

### Runtime-friendly subset

Use a smaller sample during the one-hour assignment. You can increase the sizes later.

In [ ]:

train_ds = ag_news["train"].shuffle(seed=SEED).select(
    range(min(12000, len(ag_news["train"])))
)

test_ds = ag_news["test"].shuffle(seed=SEED).select(
    range(min(3000, len(ag_news["test"])))
)

print("Train size:", len(train_ds))
print("Test size :", len(test_ds))

### Q1A - Tokenizer and tokenization

Create the DistilBERT tokenizer and a tokenization function. Tokenize both dataset splits.

**Hint:** `AutoTokenizer.from_pretrained(...)`

In [ ]:
# Q1A - Write your code here

### Q1B - Model and dynamic padding

Create:

- a `DataCollatorWithPadding`
- `id2label` and `label2id`
- `AutoModelForSequenceClassification`

Use 4 output labels.

In [ ]:
# Q1B - Write your code here

### Q1C - Metric function

Create a `compute_metrics(eval_pred)` function that returns:

- accuracy
- macro precision
- macro recall
- macro F1

**Hint:** use `precision_recall_fscore_support(..., average="macro")`.

In [ ]:
# Q1C - Write your code here

### Q1D - Fine-tune

Create `TrainingArguments` and a `Trainer`, then fine-tune DistilBERT.

Recommended starting values:

```text
learning_rate = 2e-5
batch size    = 16
2 epochs for the classroom subset
eval_strategy = "epoch"
save_strategy = "epoch"
```

You may adjust them if needed.

In [ ]:
# Q1D - Write your code here

### Q1E - Report the result

Evaluate the trained model and record:

- Accuracy:
- Macro precision:
- Macro recall:
- **Macro F1:**

Then write 2-3 sentences interpreting the result.

In [ ]:
# Q1E - Write your evaluation code here

# Q2 [10 marks] - Visualize attention with BertViz

## Task

1. Predict the test subset from Q1.
2. Find **one correct prediction**.
3. Find **one wrong prediction**.
4. Run each short example through the fine-tuned model with `output_attentions=True`.
5. Use BertViz `head_view` to inspect attention.
6. Compare what you observe.

### Important interpretation rule

An attention map shows where attention weights are concentrated. It does **not** automatically prove that a token caused the final class prediction.

### Q2A - Find one correct and one wrong example

Create predictions and identify one index where:

```text
prediction == actual label
```

and one where:

```text
prediction != actual label
```

Print the text, actual label and predicted label for both.

In [ ]:
# Q2A - Write your code here

### Q2B - Build a BertViz helper

Use:

```python
from bertviz import head_view
```

Your helper should:

1. tokenize one short text,
2. call the model with `output_attentions=True`,
3. convert IDs back to tokens,
4. pass the attention tensors and tokens to `head_view`.

**Tip:** short inputs are easier to visualize. If your Transformers version uses a faster attention backend that does not return explicit weights, switch the model to an eager attention implementation if supported.

In [ ]:
# Q2B - Write your BertViz helper here

### Q2C - Compare the attention maps

Visualize:

- the correct example
- the wrong example

Then answer:

1. Which layer/head did you inspect?
2. Which tokens received relatively high attention?
3. Was attention concentrated or spread out?
4. Did the wrong example contain ambiguous or cross-topic words?
5. What can you infer cautiously from the visualization?

In [ ]:
# Q2C - Call your visualization helper here

# Q3 [10 marks] - Compare BERT vs RoBERTa tokenization

## Task

Use these five sentences:

1. `Transformers changed NLP.`
2. `I can't believe it's already 2026!`
3. `The performance was unbelievable.`
4. `Email me at data.science@example.com.`
5. `I love AI 😊`

Compare the output of:

- `bert-base-uncased`
- `roberta-base`

For each sentence, record:

- BERT tokens
- number of BERT tokens
- RoBERTa tokens
- number of RoBERTa tokens
- one observation about the difference

In [ ]:

# Tokenizer loading - provided.
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    "FacebookAI/roberta-base"
)

sample_sentences = [
    "Transformers changed NLP.",
    "I can't believe it's already 2026!",
    "The performance was unbelievable.",
    "Email me at data.science@example.com.",
    "I love AI 😊"
]

### Q3A - Build the comparison table

Loop through the five sentences and create a DataFrame containing both tokenizations and token counts.

In [ ]:
# Q3A - Write your code here

### Q3B - Write your observations

Use your output table to answer:

1. How does BERT mark subword continuation pieces?
2. How does RoBERTa handle spaces differently?
3. Which tokenizer produces more tokens for contractions/punctuation in your examples?
4. How is the unusual character/emoji handled?
5. Why must a pretrained model use its matching tokenizer?

# Submission checklist

Before finishing, make sure your notebook contains:

- [ ] Q1 fine-tuning code
- [ ] Q1 macro F1 result
- [ ] One correct and one wrong Q2 example
- [ ] Two BertViz attention visualizations
- [ ] Written interpretation of Q2
- [ ] BERT vs RoBERTa tokenization table for all 5 sentences
- [ ] Written observations for Q3

### Marks

| Question | Marks |
|---|---:|
| Q1 - DistilBERT + macro F1 | 10 |
| Q2 - BertViz correct vs wrong | 10 |
| Q3 - BERT vs RoBERTa tokenization | 10 |
| **Total** | **30** |